# Phase 3A – Data Quality & Cleaning Challenge

## Objective
Work with intentionally messy data and apply cleaning techniques before building a pipeline. This challenge helps show why data cleaning is critical in real-world data engineering.

## Expected Effort
30–45 minutes

## Mandatory Reading
1. [Filtering Data](https://www.sparkplayground.com/tutorials/pyspark/filtering-data)
2. [Grouping Data](https://www.sparkplayground.com/tutorials/pyspark/grouping-data)

## Step 1: Create the dataset
### Load raw data
### Build messy sample data

In [0]:
data = [
    (1, "Ravi", "Hyderabad", 25),
    (2, None, "Chennai", 32),
    (None, "Arun", "Hyderabad", 28),
    (4, "Meena", None, 30),
    (4, "Meena", None, 30),
    (5, "John", "Bangalore", -5)
]

columns = ["customer_id", "name", "city", "age"]
df = spark.createDataFrame(data, columns)

df.show()
df.printSchema()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+

root
 |-- customer_id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- age: long (nullable = true)



## Step 2: Identify data issues

### Problems found in the raw dataset
- Null value in `name`
- Null value in `customer_id`
- Null value in `city`
- Duplicate row for Meena
- Invalid age value `-5`

These issues must be cleaned before aggregation, otherwise the output may be incorrect.

In [0]:
df.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+



## Step 3: Clean the data

### Cleaning rules used
- Remove rows where `customer_id`, `name`, or `city` is null
- Remove duplicate rows
- Filter rows where age is less than or equal to 0

In [0]:
clean_df = (
    df.dropna(subset=["customer_id", "name", "city"])
      .dropDuplicates()
      .filter("age > 0")
)

clean_df.show()

+-----------+----+---------+---+
|customer_id|name|     city|age|
+-----------+----+---------+---+
|          1|Ravi|Hyderabad| 25|
+-----------+----+---------+---+



## Step 4: Validate cleaning

The next step is to compare the number of rows before and after cleaning.

In [0]:
print("Raw count:", df.count())
print("Cleaned count:", clean_df.count())

Raw count: 6
Cleaned count: 1


### Validation result
- Raw count is 6 because the original dataset contains 6 rows.
- Cleaned count is 1 because rows with null keys, duplicate data, and invalid age values were removed.
- Only one fully valid row remains after cleaning.

## Step 5: Compare raw and cleaned data

In [0]:
df.show()
clean_df.show()

+-----------+-----+---------+---+
|customer_id| name|     city|age|
+-----------+-----+---------+---+
|          1| Ravi|Hyderabad| 25|
|          2| NULL|  Chennai| 32|
|       NULL| Arun|Hyderabad| 28|
|          4|Meena|     NULL| 30|
|          4|Meena|     NULL| 30|
|          5| John|Bangalore| -5|
+-----------+-----+---------+---+

+-----------+----+---------+---+
|customer_id|name|     city|age|
+-----------+----+---------+---+
|          1|Ravi|Hyderabad| 25|
+-----------+----+---------+---+



## Step 6: Perform aggregation

The final task is to calculate the number of customers per city using the cleaned dataset.

In [0]:
from pyspark.sql.functions import count
clean_df.groupBy("city").agg(count("*").alias("customer_count")).show()

+---------+--------------+
|     city|customer_count|
+---------+--------------+
|Hyderabad|             1|
+---------+--------------+



## Key Learnings
- Real-world data is messy.
- Cleaning is mandatory before processing.
- Invalid data leads to wrong results.
- Validation is essential before aggregation.

## Reflection Questions

1. What happens if cleaning is skipped?
   - Invalid rows and null values can produce incorrect results.

2. Which issue impacted results most?
   - Null keys and invalid values had the biggest impact because they made records unusable.

3. How would this affect business decisions?
   - Wrong customer counts or city-wise results could lead to bad reporting and poor decisions.

4. Can you define a cleaning checklist?
   - Check for null keys
   - Check for missing values
   - Remove duplicates
   - Filter invalid numeric values
   - Validate row counts before and after cleaning

## Conclusion

This challenge showed that raw data cannot be trusted directly. The dataset contained nulls, duplicates, and invalid values, so cleaning was required before performing aggregation. After validation, the cleaned dataset was used to calculate customers per city correctly.